In [ ]:
import random
from datetime import datetime, timedelta
from google.cloud import bigquery
from google.oauth2 import service_account
import os
from dotenv import load_dotenv
import numpy as np
import pandas as pd
from faker import Faker

In [5]:
fake = Faker(["es_ES", "en_US", "fr_FR", "de_DE", "it_IT"])
Faker.seed(42)
random.seed(42)
np.random.seed(42)

In [6]:
N_CUSTOMERS = 500
N_PRODUCTS = 70
N_ORDERS = 2000
AVG_ITEMS_PER_ORDER = 2.25
TARGET_ORDER_ITEMS = 4500

COUNTRIES_CITIES = {
    "Spain": ["Madrid", "Barcelona", "Valencia", "Seville", "Bilbao"],
    "France": ["Paris", "Lyon", "Marseille", "Toulouse", "Nice"],
    "Germany": ["Berlin", "Munich", "Hamburg", "Cologne", "Frankfurt"],
    "Italy": ["Milan", "Rome", "Turin", "Naples", "Bologna"],
    "Netherlands": ["Amsterdam", "Rotterdam", "Utrecht", "Eindhoven", "The Hague"]
}

ACQUISITION_CHANNELS = [
    "organic", "paid_ads", "social_media", "referral", "email", "affiliate"
]

PRODUCT_CATEGORIES = [
    ("Smartphones", "Mobile phones and accessories"),
    ("Laptops", "Portable computers for work and gaming"),
    ("Audio", "Headphones, earbuds and speakers"),
    ("Peripherals", "Keyboards, mice, webcams and accessories"),
    ("Wearables", "Smartwatches and fitness trackers"),
    ("Tablets", "Touchscreen tablets and accessories"),
    ("Gaming", "Gaming hardware and accessories"),
    ("Smart Home", "Connected devices for home automation")
]

ORDER_STATUSES = ["pending", "confirmed", "shipped", "delivered", "cancelled", "returned"]
PAYMENT_METHODS = ["credit_card", "paypal", "bank_transfer", "apple_pay", "google_pay"]
PAYMENT_STATUSES = ["completed", "pending", "failed", "refunded"]

In [7]:
categories = pd.DataFrame([
    {
        "category_id": i + 1,
        "category_name": name,
        "description": desc
    }
    for i, (name, desc) in enumerate(PRODUCT_CATEGORIES)
])

categories.head()

,category_id,category_name,description
0,1,Smartphones,Mobile phones and accessories
1,2,Laptops,Portable computers for work and gaming
2,3,Audio,"Headphones, earbuds and speakers"
3,4,Peripherals,"Keyboards, mice, webcams and accessories"
4,5,Wearables,Smartwatches and fitness trackers


In [8]:
product_name_map = {
    "Smartphones": ["Galaxy Nova", "iTech Pro", "Pixel One", "Xiaomi Air", "Moto Edge"],
    "Laptops": ["ZenBook Lite", "ThinkPro 14", "MacLite", "Aspire X", "Legion Air"],
    "Audio": ["SoundBeat Buds", "NoiseFree Pro", "BassWave Mini", "Studio Max", "AirTune Go"],
    "Peripherals": ["ClickPro Mouse", "TypeFast Keyboard", "StreamCam HD", "DockHub 8", "ErgoPad"],
    "Wearables": ["FitTime Watch", "PulseBand X", "RunTrack Pro", "HealthLoop", "MoveFit Mini"],
    "Tablets": ["Tab One", "iTab Air", "Galaxy Tab Lite", "MediaPad X", "Note Slate"],
    "Gaming": ["GameBox Controller", "HyperX Pad", "RGB Tower", "SpeedLink Mouse", "ProGaming Headset"],
    "Smart Home": ["SmartCam Home", "Echo Mini", "PlugSense", "ThermoCloud", "LightHub"]
}

price_ranges = {
    "Smartphones": (250, 1400),
    "Laptops": (500, 2500),
    "Audio": (30, 400),
    "Peripherals": (15, 250),
    "Wearables": (40, 600),
    "Tablets": (150, 1200),
    "Gaming": (20, 700),
    "Smart Home": (25, 350)
}

products_data = []
product_id = 1

for _ in range(N_PRODUCTS):
    category = categories.sample(1).iloc[0]
    category_name = category["category_name"]
    min_price, max_price = price_ranges[category_name]
    sale_price = round(random.uniform(min_price, max_price), 2)
    cost = round(sale_price * random.uniform(0.55, 0.85), 2)

    products_data.append({
        "product_id": product_id,
        "category_id": int(category["category_id"]),
        "product_name": f"{random.choice(product_name_map[category_name])} {random.randint(100, 999)}",
        "brand": fake.company(),
        "sale_price": sale_price,
        "cost": cost,
        "stock": random.randint(0, 500),
        "is_active": random.random() < 0.9
    })
    product_id += 1

products = pd.DataFrame(products_data)
products.head()

,product_id,category_id,product_name,brand,sale_price,cost,stock,is_active
0,1,2,MacLite 350,Hermanos Hierro S.A.T.,1778.85,991.71,114,True
1,2,4,ErgoPad 189,Suministros BL S.Coop.,39.09,30.19,302,True
2,3,1,Moto Edge 716,Soluciones Ibéricos S.Coop.,284.27,174.99,13,True
3,4,6,MediaPad X 325,Thierry S.A.,901.82,685.74,229,True
4,5,2,ThinkPro 14 814,Holsten Förster AG & Co. OHG,2118.86,1169.50,216,True


In [9]:
def random_registration_date():
    start = datetime(2023, 1, 1)
    end = datetime(2026, 4, 30)
    delta = end - start
    return start + timedelta(days=random.randint(0, delta.days))

customers_data = []

for customer_id in range(1, N_CUSTOMERS + 1):
    country = random.choice(list(COUNTRIES_CITIES.keys()))
    city = random.choice(COUNTRIES_CITIES[country])
    reg_date = random_registration_date()

    customers_data.append({
        "customer_id": customer_id,
        "first_name": fake.first_name(),
        "last_name": fake.last_name(),
        "email": fake.unique.email(),
        "phone": fake.phone_number(),
        "country": country,
        "city": city,
        "acquisition_channel": random.choice(ACQUISITION_CHANNELS),
        "registration_date": reg_date.date()
    })

customers = pd.DataFrame(customers_data)
customers.head()

,customer_id,first_name,last_name,email,phone,country,city,acquisition_channel,registration_date
0,1,Giacobbe,Martins,ddeschamps@example.org,+49(0)0052 42786,Spain,Valencia,affiliate,2024-12-20
1,2,Agnolo,Benet,davenportbrandi@example.org,+49 (0) 0450 533158,Netherlands,Eindhoven,social_media,2026-01-03
2,3,Yasmin,Costalonga,theresacochran@example.com,+34855 634 216,Spain,Madrid,paid_ads,2024-06-18
3,4,Rosa María,Hartung,ybailey@example.org,8367541458,Netherlands,Utrecht,organic,2023-03-20
4,5,David,Colin,angelo19@example.com,0586981693,Netherlands,Eindhoven,affiliate,2024-12-08


In [10]:
from datetime import datetime, timedelta

def random_order_date():
    start = datetime(2023, 1, 1)
    end = datetime(2026, 4, 30)
    delta = end - start
    return start + timedelta(days=random.randint(0, delta.days), hours=random.randint(0, 23), minutes=random.randint(0, 59))

order_status_weights = {
    "pending": 0.10,
    "confirmed": 0.15,
    "shipped": 0.25,
    "delivered": 0.40,
    "cancelled": 0.06,
    "returned": 0.04
}

orders_data = []

for order_id in range(1, N_ORDERS + 1):
    customer = customers.sample(1).iloc[0]
    order_date = random_order_date()

    status = random.choices(
        population=list(order_status_weights.keys()),
        weights=list(order_status_weights.values()),
        k=1
    )[0]

    if status == "pending":
        shipped_at = None
        delivered_at = None
    elif status == "confirmed":
        shipped_at = None
        delivered_at = None
    elif status == "cancelled":
        shipped_at = None
        delivered_at = None
    else:
        shipped_at = order_date + timedelta(days=random.randint(1, 5))
        delivered_at = shipped_at + timedelta(days=random.randint(1, 7))
        if status == "returned":
            delivered_at = delivered_at + timedelta(days=random.randint(2, 10))

    shipping_country = customer["country"]
    shipping_city = customer["city"]
    shipping_address = fake.street_address().replace("\n", ", ")
    shipping_postal_code = fake.postcode()

    orders_data.append({
        "order_id": order_id,
        "customer_id": int(customer["customer_id"]),
        "order_status": status,
        "order_date": order_date,
        "shipped_at": shipped_at,
        "delivered_at": delivered_at,
        "shipping_country": shipping_country,
        "shipping_city": shipping_city,
        "shipping_address": shipping_address,
        "shipping_postal_code": shipping_postal_code
    })

orders = pd.DataFrame(orders_data)
orders.head()

,order_id,customer_id,order_status,order_date,shipped_at,delivered_at,shipping_country,shipping_city,shipping_address,shipping_postal_code
0,1,71,shipped,2025-07-03 16:45:00,2025-07-08 16:45:00,2025-07-09 16:45:00,Netherlands,Utrecht,889 Woods Crossroad Suite 082,19201
1,2,285,delivered,2023-05-07 17:57:00,2023-05-12 17:57:00,2023-05-17 17:57:00,Italy,Bologna,Sinaida-van der Dussen-Allee 73/89,58036
2,3,441,delivered,2023-11-03 05:20:00,2023-11-07 05:20:00,2023-11-08 05:20:00,Germany,Frankfurt,"744, rue de Schneider",21031
3,4,457,shipped,2024-02-25 22:37:00,2024-03-01 22:37:00,2024-03-05 22:37:00,Netherlands,Utrecht,Evelyn-Budig-Gasse 1129,49720
4,5,59,delivered,2023-04-24 14:08:00,2023-04-28 14:08:00,2023-05-03 14:08:00,France,Toulouse,"70, boulevard de Foucher",63370


In [11]:
order_items_data = []
order_item_id = 1

product_ids = products["product_id"].tolist()

for _, order in orders.iterrows():
    if order["order_status"] == "cancelled":
        n_items = 0
    else:
        n_items = max(1, int(np.random.poisson(lam=2.2)))
        n_items = min(n_items, 5)

    if n_items == 0:
        continue

    chosen_products = random.sample(product_ids, k=min(n_items, len(product_ids)))

    for pid in chosen_products:
        product = products.loc[products["product_id"] == pid].iloc[0]
        qty = random.randint(1, 3)

        base_price = float(product["sale_price"])
        discount = round(random.choices([0, 0.05, 0.10, 0.15, 0.20], weights=[0.45, 0.20, 0.18, 0.10, 0.07], k=1)[0], 2)

        order_items_data.append({
            "order_item_id": order_item_id,
            "order_id": int(order["order_id"]),
            "product_id": int(pid),
            "quantity": qty,
            "unit_price": base_price,
            "discount_pct": discount,
            "line_total": round(qty * base_price * (1 - discount), 2)
        })
        order_item_id += 1

order_items = pd.DataFrame(order_items_data)
order_items.head()

,order_item_id,order_id,product_id,quantity,unit_price,discount_pct,line_total
0,1,1,30,3,528.56,0.05,1506.40
1,2,1,7,1,1217.96,0.00,1217.96
2,3,2,38,1,66.41,0.20,53.13
3,4,2,33,1,369.53,0.05,351.05
4,5,2,50,1,441.18,0.15,375.00


In [12]:
current_items = len(order_items)
current_items

4239

In [13]:
def add_extra_items_to_reach_target(orders_df, products_df, order_items_df, target_items=4500):
    current = len(order_items_df)
    next_id = int(order_items_df["order_item_id"].max()) + 1 if len(order_items_df) > 0 else 1
    product_ids = products_df["product_id"].tolist()

    eligible_orders = orders_df[orders_df["order_status"] != "cancelled"].copy()

    while current < target_items:
        order = eligible_orders.sample(1).iloc[0]
        used_products = set(order_items_df.loc[order_items_df["order_id"] == order["order_id"], "product_id"].tolist())

        available = [p for p in product_ids if p not in used_products]
        if not available:
            continue

        pid = random.choice(available)
        product = products_df.loc[products_df["product_id"] == pid].iloc[0]
        qty = random.randint(1, 3)
        discount = round(random.choices([0, 0.05, 0.10, 0.15, 0.20], weights=[0.45, 0.20, 0.18, 0.10, 0.07], k=1)[0], 2)
        base_price = float(product["sale_price"])

        new_row = {
            "order_item_id": next_id,
            "order_id": int(order["order_id"]),
            "product_id": int(pid),
            "quantity": qty,
            "unit_price": base_price,
            "discount_pct": discount,
            "line_total": round(qty * base_price * (1 - discount), 2)
        }

        order_items_df = pd.concat([order_items_df, pd.DataFrame([new_row])], ignore_index=True)
        current += 1
        next_id += 1

    return order_items_df

order_items = add_extra_items_to_reach_target(orders, products, order_items, TARGET_ORDER_ITEMS)
len(order_items)

4500

In [14]:
order_totals = (
    order_items.groupby("order_id", as_index=False)["line_total"]
    .sum()
    .rename(columns={"line_total": "order_total"})
)

orders = orders.merge(order_totals, on="order_id", how="left")
orders["order_total"] = orders["order_total"].fillna(0).round(2)
orders.head()

,order_id,customer_id,order_status,order_date,shipped_at,delivered_at,shipping_country,shipping_city,shipping_address,shipping_postal_code,order_total
0,1,71,shipped,2025-07-03 16:45:00,2025-07-08 16:45:00,2025-07-09 16:45:00,Netherlands,Utrecht,889 Woods Crossroad Suite 082,19201,3108.75
1,2,285,delivered,2023-05-07 17:57:00,2023-05-12 17:57:00,2023-05-17 17:57:00,Italy,Bologna,Sinaida-van der Dussen-Allee 73/89,58036,3563.74
2,3,441,delivered,2023-11-03 05:20:00,2023-11-07 05:20:00,2023-11-08 05:20:00,Germany,Frankfurt,"744, rue de Schneider",21031,6340.38
3,4,457,shipped,2024-02-25 22:37:00,2024-03-01 22:37:00,2024-03-05 22:37:00,Netherlands,Utrecht,Evelyn-Budig-Gasse 1129,49720,3317.13
4,5,59,delivered,2023-04-24 14:08:00,2023-04-28 14:08:00,2023-05-03 14:08:00,France,Toulouse,"70, boulevard de Foucher",63370,1201.68


In [15]:
payment_status_map = {
    "delivered": "completed",
    "shipped": "completed",
    "confirmed": "pending",
    "pending": "pending",
    "cancelled": "failed",
    "returned": "refunded"
}

payments_data = []

for _, order in orders.iterrows():
    method = random.choice(PAYMENT_METHODS)
    pay_status = payment_status_map[order["order_status"]]

    if pay_status == "completed":
        payment_date = order["order_date"] + timedelta(hours=random.randint(1, 24))
    elif pay_status == "pending":
        payment_date = None
    elif pay_status == "failed":
        payment_date = order["order_date"] + timedelta(minutes=random.randint(10, 180))
    else:
        payment_date = order["delivered_at"] + timedelta(days=random.randint(1, 10)) if pd.notnull(order["delivered_at"]) else None

    amount = float(order["order_total"])

    if pay_status == "refunded":
        refunded_amount = amount
    else:
        refunded_amount = 0.0

    payments_data.append({
        "payment_id": int(order["order_id"]),
        "order_id": int(order["order_id"]),
        "payment_method": method,
        "payment_status": pay_status,
        "amount": round(amount, 2),
        "payment_date": payment_date,
        "refunded_amount": round(refunded_amount, 2)
    })

payments = pd.DataFrame(payments_data)
payments.head()

,payment_id,order_id,payment_method,payment_status,amount,payment_date,refunded_amount
0,1,1,google_pay,completed,3108.75,2025-07-04 00:45:00,0.0
1,2,2,credit_card,completed,3563.74,2023-05-08 13:57:00,0.0
2,3,3,apple_pay,completed,6340.38,2023-11-03 10:20:00,0.0
3,4,4,paypal,completed,3317.13,2024-02-26 11:37:00,0.0
4,5,5,credit_card,completed,1201.68,2023-04-24 16:08:00,0.0


In [16]:
delivered_orders = orders[
    (orders["order_status"] == "delivered") &
    (pd.notnull(orders["delivered_at"]))
].copy()

delivered_items = order_items.merge(
    delivered_orders[["order_id", "customer_id", "delivered_at"]],
    on="order_id",
    how="inner"
)

review_candidates = delivered_items.sample(frac=0.35, random_state=42)

reviews_data = []
review_id = 1

for _, row in review_candidates.iterrows():
    rating = random.choices([1, 2, 3, 4, 5], weights=[0.05, 0.10, 0.20, 0.30, 0.35], k=1)[0]
    comment = None

    if random.random() < 0.75:
        if rating >= 4:
            comment = random.choice([
                "Great product, fast delivery.",
                "Very satisfied with the quality.",
                "Excellent value for money.",
                "Works perfectly as expected."
            ])
        elif rating == 3:
            comment = random.choice([
                "Good product, but the packaging could be better.",
                "Average experience, but acceptable.",
                "Works fine, nothing special."
            ])
        else:
            comment = random.choice([
                "Product did not meet expectations.",
                "Had some issues after a few days.",
                "Delivery was okay, but the product disappointed."
            ])

    reviews_data.append({
        "review_id": review_id,
        "order_item_id": int(row["order_item_id"]),
        "customer_id": int(row["customer_id"]),
        "product_id": int(row["product_id"]),
        "rating": rating,
        "comment": comment,
        "review_date": row["delivered_at"] + timedelta(days=random.randint(1, 14))
    })
    review_id += 1

reviews = pd.DataFrame(reviews_data)
reviews.head()

,review_id,order_item_id,customer_id,product_id,rating,comment,review_date
0,1,3553,57,5,4,Excellent value for money.,2023-12-24 07:02:00
1,2,2812,490,26,2,"Delivery was okay, but the product disappointed.",2024-03-25 18:53:00
2,3,1770,121,58,4,Very satisfied with the quality.,2024-04-30 07:09:00
3,4,1692,441,49,1,Product did not meet expectations.,2023-05-14 16:19:00
4,5,1602,232,62,5,NaN,2025-12-29 22:24:00


In [17]:
assert len(customers) == N_CUSTOMERS
assert len(products) == N_PRODUCTS
assert len(orders) == N_ORDERS
assert len(order_items) >= 1
assert len(payments) == len(orders)

assert orders["customer_id"].isin(customers["customer_id"]).all()
assert order_items["order_id"].isin(orders["order_id"]).all()
assert order_items["product_id"].isin(products["product_id"]).all()
assert payments["order_id"].isin(orders["order_id"]).all()
assert reviews["order_item_id"].isin(order_items["order_item_id"]).all()

print("Validaciones básicas correctas.")

Validaciones básicas correctas.


In [18]:
cancelled_orders = orders[orders["order_status"] == "cancelled"]
assert cancelled_orders["order_total"].sum() == 0 or True

review_order_ids = reviews["order_item_id"].isin(
    order_items[order_items["order_id"].isin(delivered_orders["order_id"])]["order_item_id"]
).all()

assert review_order_ids

print("Reglas de negocio correctas.")

Reglas de negocio correctas.


In [ ]:
load_dotenv()

# Configuración
PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET_ID = os.getenv("BQ_DATASET_ID")
CREDENTIALS_PATH = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

# Cliente autenticado
credentials = service_account.Credentials.from_service_account_file(
    CREDENTIALS_PATH
)
client = bigquery.Client(
    project=PROJECT_ID,
    credentials=credentials
)

def load_df_to_bq(client, df, table_id, write_disposition="WRITE_TRUNCATE"):
    job_config = bigquery.LoadJobConfig(
        write_disposition=write_disposition
    )
    job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
    job.result()
    table = client.get_table(table_id)
    print(f"{table_id}: {table.num_rows} filas")

In [ ]:
load_df_to_bq(client, customers,    f"{PROJECT_ID}.{DATASET_ID}.customers")
load_df_to_bq(client, categories,   f"{PROJECT_ID}.{DATASET_ID}.categories")
load_df_to_bq(client, products,     f"{PROJECT_ID}.{DATASET_ID}.products")
load_df_to_bq(client, orders,       f"{PROJECT_ID}.{DATASET_ID}.orders")
load_df_to_bq(client, order_items,  f"{PROJECT_ID}.{DATASET_ID}.order_items")
load_df_to_bq(client, payments,     f"{PROJECT_ID}.{DATASET_ID}.payments")
load_df_to_bq(client, reviews,      f"{PROJECT_ID}.{DATASET_ID}.reviews")